# Notebook 2: Predicting PoC output

## Blind-test methodology

We ask a model "What does this code output? Respond with ONLY the output.", compare its
prediction to the interpreter output, and report exact-match accuracy. The same test on
equivalent Python programs is the control; the gap is the differential opacity.

This notebook runs the methodology on a small corpus using a local Qwen-Coder-7B via `mlx_lm`.
Cells that call the model are marked and can be skipped if `mlx_lm` is not installed.

In [ ]:
# Setup
import sys, os, subprocess, tempfile, json
sys.path.insert(0, os.path.abspath('..'))

PROJECT_ROOT = os.path.abspath('..')
INTERPRETER  = os.path.join(PROJECT_ROOT, 'poc_v2.py')
CORPUS_DIR   = os.path.join(PROJECT_ROOT, 'experiments', 'poc_programs')
print('Interpreter:', INTERPRETER)
print('Corpus dir :', CORPUS_DIR)

In [ ]:
# ── Load 5 corpus snippets and record their ground-truth outputs ──────────────

def run_poc(code: str) -> str:
    """Execute a PoC snippet and return its stdout."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.poc', delete=False) as f:
        f.write(code)
        path = f.name
    result = subprocess.run(
        [sys.executable, INTERPRETER, path],
        capture_output=True, text=True, timeout=10
    )
    os.unlink(path)
    return result.stdout.strip()

# Read 5 files from corpus
corpus = []
for idx in range(5):
    fname = os.path.join(CORPUS_DIR, f'basic_{idx:03d}.poc')
    with open(fname) as fh:
        code = fh.read().strip()
    expected = run_poc(code)
    corpus.append({'id': idx, 'code': code, 'expected': expected})

print(f'Loaded {len(corpus)} snippets\n')
for item in corpus:
    print(f'--- Snippet {item["id"]} ---')
    print(item['code'])
    print(f'Expected output: {repr(item["expected"])}')
    print()

In [ ]:
# ── [REQUIRES mlx_lm] Query Qwen-Coder-7B on one PoC snippet ────────────────
# Skip this cell if mlx_lm is not installed.

try:
    from mlx_lm import load, generate
    MLX_AVAILABLE = True
except ImportError:
    MLX_AVAILABLE = False
    print('mlx_lm not installed — skipping model inference cells.')
    print('Install with: pip install mlx-lm')

if MLX_AVAILABLE:
    print('Loading Qwen/Qwen2.5-Coder-7B-Instruct ...')
    model, tokenizer = load('Qwen/Qwen2.5-Coder-7B-Instruct')
    print('Model loaded.')

    # Use the sum-1-to-5 snippet (index 0 = basic_000, but let's use a richer one)
    poc_code = '''yield total = 0
yield i = 1
finally i <= 5:
    case total = total + i
    case i = i + 1
break(total)'''

    prompt = (
        'What does this code output? '
        'Respond with ONLY the output, nothing else.\n\n'
        f'```\n{poc_code}\n```'
    )
    messages = [{'role': 'user', 'content': prompt}]
    chat = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    response = generate(model, tokenizer, prompt=chat, max_tokens=100, verbose=False)

    print('\n=== PoC snippet ===')
    print(poc_code)
    print('\nModel says:', repr(response.strip()))
    print('Actual output: 15')
    print('Correct?', response.strip() == '15')

In [ ]:
# ── [REQUIRES mlx_lm] Same logic, but in plain Python — model aces it ────────

if MLX_AVAILABLE:
    python_equiv = '''total = 0
i = 1
while i <= 5:
    total = total + i
    i = i + 1
print(total)'''

    prompt_py = (
        'What does this Python code output? '
        'Respond with ONLY the output, nothing else.\n\n'
        f'```python\n{python_equiv}\n```'
    )
    messages_py = [{'role': 'user', 'content': prompt_py}]
    chat_py = tokenizer.apply_chat_template(
        messages_py, tokenize=False, add_generation_prompt=True
    )
    response_py = generate(model, tokenizer, prompt=chat_py, max_tokens=100, verbose=False)

    print('=== Python equivalent ===')
    print(python_equiv)
    print('\nModel says:', repr(response_py.strip()))
    print('Actual output: 15')
    print('Correct?', response_py.strip() == '15')

In [ ]:
# ── [REQUIRES mlx_lm] Run all 5 corpus snippets and tally the score ───────────

if MLX_AVAILABLE:
    correct = 0
    results = []

    for item in corpus:
        prompt = (
            'What does this code output? '
            'Respond with ONLY the output, nothing else.\n\n'
            f'```\n{item["code"]}\n```'
        )
        msgs = [{'role': 'user', 'content': prompt}]
        chat = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        pred = generate(model, tokenizer, prompt=chat, max_tokens=80, verbose=False).strip()
        hit  = pred == item['expected']
        correct += int(hit)
        results.append({'id': item['id'], 'expected': item['expected'], 'predicted': pred, 'correct': hit})
        print(f'Snippet {item["id"]}: expected={repr(item["expected"])}  got={repr(pred)}  ✓' if hit
              else f'Snippet {item["id"]}: expected={repr(item["expected"])}  got={repr(pred)}  ✗')

    print(f'\nScore: {correct}/{len(corpus)} = {correct/len(corpus):.0%}')
else:
    # Show pre-computed illustrative results
    print('(mlx_lm not available — showing illustrative results from the paper)')
    illustrative = [
        {'id': 0, 'expected': '0',       'predicted': 'None',   'correct': False},
        {'id': 1, 'expected': '3',       'predicted': '3',      'correct': True},
        {'id': 2, 'expected': '2',       'predicted': '2',      'correct': True},
        {'id': 3, 'expected': 'true',    'predicted': 'True',   'correct': False},
        {'id': 4, 'expected': 'true',    'predicted': 'True',   'correct': False},
    ]
    for r in illustrative:
        mark = '✓' if r['correct'] else '✗'
        print(f'Snippet {r["id"]}: expected={repr(r["expected"])}  got={repr(r["predicted"])}  {mark}')
    correct = sum(r['correct'] for r in illustrative)
    print(f'\nScore: {correct}/{len(illustrative)} = {correct/len(illustrative):.0%}')

## Results

Full experiment (11 models, 40 corpus programs):

- Mean PoC accuracy: 18% (vs. 75–100% on equivalent Python)
- GPT-4o-mini: 75% Python → 0% PoC
- `async`, `await`, `del` reached 100% confusion across all models tested

See Notebook 4 for full results and plots.